<a href="https://colab.research.google.com/github/HenriqueGau/WebSite/blob/main/un_base_de_dados1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
# ...........Passos..................
# 1. Aquisição de Dados
# 2. Processamento de Dados
# 3. Análise e Manipulação de Dados
# 4. Exibição dos Dados
#....................................
# -------------------------------------------------------------------------------------------------

# 1. Aquisição de Dados (Parte 1)

# -------------------------------------------------------------------------------------------------

#-------------!!Código feito com base no Google.Colab resources!!---------------------------------

# Código para ler arquivo CSV ou Excel e acrescentar a base de dados cada pessoa.
# Importando Biblioteca e Recursos.

import pandas as pd
from datetime import datetime
from google.colab import files
import io
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import folium
from folium.plugins import HeatMap

# ----------------------

# Colunas esperadas

# ----------------------

# Vetor de colunas esperadas

EXPECTED_COLS = [
    "CPF", "Nome", "Data_Nascimento", "Bairro Casa", "Bairro Trabalho",
    "N_Pessoas_Casa", "Contaminado", "Familiar_Contaminado",
    "Tempo_Em_Casa", "Tempo_Fora_Casa", "Salario",
    "Bairro Intermediario", "Data_Envio"#, "Amostra", "Bairro"
]

# -----------------------

# Lista de Bairros : Opções

# -----------------------

UBATUBA_BAIRROS = [
    "Centro", "Itaguá", "Praia Grande", "Toninhas", "Enseada",
    "Lazaro", "Domenico", "Estufa II", "Rio Escuro", "Perequê-Açu",
    "Sertão da Quina", "Maranduba", "Sapê", "Tabatinga", "Cunhambebe",
    "Mata Atlântica", "Rio da Prata", "Folha Seca", "Figueira", "Sesmaria", "Ipiranguinha", "Taquaral", "Bela Vista",
    "Jardim Carolina", "Ressaca", "Pedreira", "Semidouro", "Horto", "Monte Valério"
]

UBATUBA_BAIRROS.sort()

# .sort(): ordenar a lista em ordem alfabetica

# Dicionário associando bairros a latitudes e longitudes (copiado de cell e0c02dce)

bairros_coords = {
    "Centro": {"latitude": -23.435556, "longitude": -45.073889},
    "Itaguá": {"latitude": -23.455000, "longitude": -45.066667},
    "Praia Grande": {"latitude": -23.467222, "longitude": -45.065000},
    "Toninhas": {"latitude": -23.487222, "longitude": -45.075833},
    "Enseada": {"latitude": -23.491389, "longitude": -45.091111},
    "Lazaro": {"latitude": -23.501944, "longitude": -45.132500},
    "Domenico": {"latitude": -23.497500, "longitude": -45.141667},
    "Estufa II": {"latitude": -23.455556, "longitude": -45.080833},
    "Rio Escuro": {"latitude": -23.487778, "longitude": -45.154167},
    "Perequê-Açu": {"latitude": -23.414444, "longitude": -45.063333},
    "Sertão da Quina": {"latitude": -23.530000, "longitude": -45.243333},
    "Maranduba": {"latitude": -23.538889, "longitude": -45.234722},
    "Sapê": {"latitude": -23.528056, "longitude": -45.229167},
    "Tabatinga": {"latitude": -23.569722, "longitude": -45.268611},
    "Cunhambebe": {"latitude": -22.961389, "longitude": -44.440000},
    "Mata Atlântica": {"latitude": -23.456944, "longitude": -45.117500},
    "Rio da Prata": {"latitude": -23.556389, "longitude": -45.248333},
    "Folha Seca": {"latitude": -23.482222, "longitude": -45.166111},
    "Figueira": {"latitude": -23.396111, "longitude": -45.123889},
    "Sesmaria": {"latitude": -23.467500, "longitude": -45.090000},
    "Ipiranguinha": {"latitude": -23.428333, "longitude": -45.110833},
    "Taquaral": {"latitude": -23.406111, "longitude": -45.065000},
    "Bela Vista": {"latitude": -23.442500, "longitude": -45.100278},
    "Jardim Carolina": {"latitude": -23.437778, "longitude": -45.090833},
    "Ressaca": {"latitude": -23.429722, "longitude": -45.089722},
    "Pedreira": {"latitude": -23.419722, "longitude": -45.072222},
    "Semidouro": {"latitude": -23.408889, "longitude": -45.066111},
    "Horto": {"latitude": -23.421111, "longitude": -45.108056},
    "Monte Valério": {"latitude": -23.452500, "longitude": -45.106944},
}

# Converter para DataFrame para melhor visualização e uso no merge(Junção)

zoom_level=13

bairros_coords_df = pd.DataFrame.from_dict(bairros_coords, orient='index')
bairros_coords_df.index.name = 'Bairro'

# --------------------------

# BASE GLOBAL (DataFrame)

# --------------------------

if 'base_df' not in globals():
    base_df = pd.DataFrame(columns=EXPECTED_COLS)
# Removed the else if block that printed the review message
# elif not base_df.empty:
#     print('Revise o arquivos, pode estar faltando dados ou contendo dados errados. \n Se tudo estiver correto, o formato de colunas do arquivo está incorreto')

# DataFrame temporário para armazenar os dados adicionados na sessão manual atual (Últimos selecionados)

if 'session_manual_df' not in globals():
    session_manual_df = pd.DataFrame(columns=EXPECTED_COLS)

# --------------------------

# FUNÇÕES AUXILIARES

# --------------------------

#Garante que o DataFrame tem as colunas esperadas (adiciona NaNs para faltantes).

def ensure_columns(df):
    for c in EXPECTED_COLS:
        if c not in df.columns:
            df[c] = pd.NA
    # reordenar colunas para consistência
    return df[EXPECTED_COLS]

# Adiciona um DataFrame (df_new) à base global, após normalizar colunas.

def add_from_dataframe(df_new):
    global base_df
    with output_area:
        mostrar_base()
        # Removed the call to generate_density_heatmap here
        # generate_density_heatmap(base_df, bairros_coords_df) # Call heatmap generation
        display(widgets.VBox([btn_view_all, btn_back_to_start, btn_delete_all])) # Add delete button

# Lê conteúdo enviado e retorna DataFrame.

def parse_uploaded_filecontent(filename, content_bytes):
    try:
        if filename.lower().endswith('.csv'):
            return pd.read_csv(io.BytesIO(content_bytes))
        elif filename.lower().endswith(('.xls', '.xlsx')):
            return pd.read_excel(io.BytesIO(content_bytes))
        else:
            raise ValueError("Formato de arquivo sem suporte. Use os formatos .csv ou .xlsx/.xls")
    except Exception as e:
        raise

def mostrar_base(df_to_show=None, title="Base de entrevistados (atual)"):
    """Exibe um DataFrame e retorna um widget HTML."""
    df_display = df_to_show if df_to_show is not None else base_df
    html_output = widgets.HTML(f"<h3> {title}</h3>")
    if len(df_display) == 0:
        html_output = widgets.HTML("<b> Base vazia.</b>")
    else:
        # Add the FutureWarning fix and return HTML widget
        html_output = widgets.HTML(df_display.astype(object).fillna("").infer_objects(copy=False).to_html())
    return html_output # Return the HTML widget


# Removed the generate_density_heatmap function definition here
# Genera e exibe um heatmap da densidade populacional por bairro.
# Argumentos:
        # df (pd.DataFrame): O DataFrame contendo os dados dos entrevistados.
        # bairros_coords_df (pd.DataFrame): O DataFrame com as coordenadas dos bairros

# def generate_density_heatmap(df, bairros_coords_df):
#     if df.empty:
#         print("Base de dados vazia. Não é possível gerar o heatmap(Mapa de Calor). Adicione dados à Base de Dados e retorne.")
#         return

#     # 1. Agregar dados por bairro (Apesar de não exato)
#     # Usar 'Bairro Casa' para a densidade populacional.

#     densidade_bairros_df = df.groupby('Bairro Casa')['N_Pessoas_Casa'].sum().reset_index()
#     densidade_bairros_df = densidade_bairros_df.rename(columns={'N_Pessoas_Casa': 'Total_Pessoas'})

#     # Garantir que os nomes de Bairros são strings and stripped for accurate merging

#     densidade_bairros_df['Bairro Casa'] = densidade_bairros_df['Bairro Casa'].astype(str).str.strip()
#     bairros_coords_df.index = bairros_coords_df.index.astype(str).str.strip()

#     # 2. Combinar com coordenadas
#     # Certificar-se de que o índice de bairros_coords_df é o nome do bairro (Têm que estar na mesma ordem, eu acho.)

#     if bairros_coords_df.index.name != 'Bairro':
#          bairros_coords_df = bairros_coords_df.copy()
#          bairros_coords_df.index.name = 'Bairro'

#     heatmap_data = pd.merge(densidade_bairros_df, bairros_coords_df, left_on='Bairro Casa', right_index=True, how='left')

#     # Reportar bairros que não foram encontrados um correspondente.

#     missing_coords = heatmap_data[heatmap_data['latitude'].isna()]['Bairro Casa'].tolist()
#     if missing_coords:
#         print(f"Aviso: Não foram encontradas coordenadas para os seguintes bairros: {', '.join(missing_coords)}")

#     # 3. Preparar dados para o heatmap(Mapa de Calor)
#     # Checar e converter tipos de dados se necessario.

#     heatmap_data['Total_Pessoas'] = pd.to_numeric(heatmap_data['Total_Pessoas'], errors='coerce')

#     # Identificar coordenadas e tamanhos de colunas.

#     latitude_col = 'latitude'
#     longitude_col = 'longitude'
#     weight_col = 'Total_Pessoas'

#     # Elimine as linhas com valores ausentes nas colunas necessárias.
#     # Considerar bairros com 0 pessoas como pontos de dados válidos com peso 0. (Validação de bairros sem entrevistados ou dados analisadp)

#     heatmap_data_cleaned = heatmap_data.dropna(subset=[latitude_col, longitude_col]).copy()
#     heatmap_data_cleaned[weight_col] = heatmap_data_cleaned[weight_col].fillna(0) # Preencher missing weights com 0

#     # Cria a lista de pontos de dados para o mapa de calor [latitude, longitude, peso].

#     heatmap_points = heatmap_data_cleaned[[latitude_col, longitude_col, weight_col]].values.tolist()

#     if not heatmap_points:
#         print("Sem dados válidos com coordenadas para gerar o heatmap.")
#         return

#     # Approximate central coordinates for Ubatuba
#     ubatuba_coords = [-23.4500, -45.0900]

#     # zoom_level = 13
#     m = folium.Map(location=ubatuba_coords, zoom_start=zoom_level)

#     # Add the heatmap layer
#     HeatMap(heatmap_points).add_to(m)

#     # ----- JA EXIBIDO EM OUTRA CÉLULA -----

#     # 5. Exibir o mapa
#     # print("Mapa de Calor da Densidade Populacional:")
#     # display(m)

# --------------------------------------------------

# Funcoes para Manipulações: Ações da interface do usuário

# --------------------------------------------------

# Abre diálogo nativo de upload (files.upload). Lê automaticamente e adiciona.

def on_click_upload(_):
    with output_area:
        clear_output(wait=True)
        display(widgets.HTML("<b> Abra o diálogo e selecione seu arquivo CSV ou Excel.</b>"))
        uploaded = files.upload()
        if not uploaded:
            display(widgets.HTML("<b> Nenhum arquivo enviado.</b>"))
            display(ui_box)
            return

        # processar cada arquivo (normalmente só 1 com todos os dados)

        for filename, fileinfo in uploaded.items():
            try:
                df_new = parse_uploaded_filecontent(filename, fileinfo['content']) # Use fileinfo['content']
            except Exception as e:
                display(widgets.HTML(f"<b style='color:red'> Erro ao ler {filename}: {e}</b>"))
                continue

            # converter datas se existir coluna parecida
            for col in df_new.columns:
                if 'data' in col.lower() or 'envio' in col.lower() or 'nascimento' in col.lower():
                    try:
                        df_new[col] = pd.to_datetime(df_new[col]).dt.date
                    except:
                        pass

            # Mapear colunas simples por similaridade (se for necessário)
            # Aqui assumimos que o usuário usará nomes compatíveis; se quiser, podemos adicionar mapeamento automático

            add_from_dataframe(df_new)


# Coletando os valores do formulário manual e adicionando à base temporária da sessão.
def on_submit_manual(_):
    global session_manual_df
    # Basic validation e conversão do tipo de dado
    cpf_value = frm_cpf.value.strip()

    with output_area:
        clear_output(wait=True)  # Clear previous output
        # Simple check para numeros de CPF
        if not cpf_value.isdigit() or len(cpf_value) != 11:
             display(widgets.HTML("<b style='color:red'>Erro: CPF deve conter 11 dígitos numéricos.</b>"))
             display(manual_form) # Reaparecer o formulario
             display(btn_delete_all) # Adicionar botão delete
             return # Stop processing if validation falhar

        dados = {
            "CPF": cpf_value,
            "Nome": frm_nome.value.strip(),
            "Data_Nascimento": frm_data_nasc.value.strftime("%Y-%m-%d") if frm_data_nasc.value else pd.NA,
            "Bairro Casa": frm_bairro_casa.value, # Usando o valor do Dropdown
            "Bairro Trabalho": frm_bairro_trabajo.value, # Usando o valor do Dropdown
            "N_Pessoas_Casa": frm_n_pessoas.value,
            "Contaminado": frm_contaminado.value,
            "Familiar_Contaminado": frm_familiar_cont.value,
            "Tempo_Em_Casa": frm_tempo_casa.value,
            "Tempo_Fora_Casa": frm_tempo_fora.value,
            "Salario": frm_salario.value,
            "Bairro Intermediario": frm_bairro_trajeto.value, # Usando o valor do Dropdown
            "Data_Envio": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
           # "Amostra": frm_amostra.value,
           # "Bairro": frm_bairro.value.strip()
        }

        # transformar em DataFrame(Conjunto de Dados) e adicionar à base de dados.
        df_temp = pd.DataFrame([dados])
        df_temp = ensure_columns(df_temp)

        # Adicionar aos Conjuntos de dados(Dataframe) da sessão manual
        session_manual_df = pd.concat([session_manual_df, df_temp], ignore_index=True)

        # Create a VBox containing the session data, success message, and buttons
        session_feedback_box = widgets.VBox([
            widgets.HTML("<b>Entrevistado adicionado à base temporária da sessão. Clique em 'Preencher novo entrevistado' para adicionar mais ou 'Finalizar' para adicionar à base principal.</b>"),
            mostrar_base(df_to_show=session_manual_df, title="Entrevistados adicionados nesta sessão (Temporário)"), # Use the returned widget
            widgets.VBox([btn_add_more, btn_finalize, btn_view_all, btn_back_to_start, btn_delete_all])
        ])

        # Display the combined VBox
        display(session_feedback_box)


# Voltar para o formulário manual para adicionar mais.

def on_click_add_more(_):
    with output_area:
        clear_output(wait=True)
        display(widgets.HTML("<h4>Formulário manual</h4><b>Preencha os campos e clique em 'Enviar entrevistado'.</b>"))
        display(manual_form) # Mostrar o formulário
        reset_manual_form() # Limpar campos do formulário
        display(btn_delete_all) # Add delete button

# Finaliza a adição manual e adiciona os dados da sessão à base global.

def on_click_finalize(_):
    global base_df, session_manual_df
    rows_added_this_session = len(session_manual_df)
    if rows_added_this_session > 0:
        base_df = pd.concat([base_df, session_manual_df], ignore_index=True)
        with output_area:
            clear_output(wait=True)
            display(widgets.HTML("<b> Adição manual finalizada.</b>"))
            # Get the tail of base_df containing the newly added rows
            newly_added_df = base_df.tail(rows_added_this_session)
            mostrar_base(df_to_show=newly_added_df, title="Entrevistados adicionados nesta sessão")
            # Removed the call to generate_density_heatmap here
            # generate_density_heatmap(base_df, bairros_coords_df) # Chamar a geração do mapa de calor
    else:
        with output_area:
            clear_output(wait=True)
            display(widgets.HTML("<b> Nenhum entrevistado adicionado nesta sessão.</b>"))

    session_manual_df = pd.DataFrame(columns=EXPECTED_COLS) # Limpar DataFrame da sessão
    with output_area:
         display(widgets.VBox([btn_view_all, btn_back_to_start, btn_delete_all])) # Add delete all button

# Mostra a base de dados completa.

def on_click_view_all(_):
    with output_area:
        clear_output(wait=True)
        mostrar_base(df_to_show=base_df, title="Base de dados completa")
        # Removed the call to generate_density_heatmap here
        # generate_density_heatmap(base_df, bairros_coords_df) # Call heatmap generation
        display(widgets.VBox([btn_back_to_start, btn_delete_all])) # Add delete button

def on_click_back_to_start(_):
    """Volta para a escolha inicial (Subir arquivo ou Adicionar manualmente)."""
    with output_area:
        clear_output(wait=True)
        display(output_info) # Exibe a mensagem inicial
        display(choice) # Exibe os botões de escolha
        on_choice_change({'new': choice.value}) # Redraw the appropriate UI based on the current choice

def on_click_delete_all(_):
    """Deleta toda a base de dados."""
    global base_df
    base_df = pd.DataFrame(columns=EXPECTED_COLS)
    with output_area:
        clear_output(wait=True)
        display(widgets.HTML("<b>Base de dados deletada.</b>"))
        mostrar_base() # Show empty base
        display(widgets.VBox([btn_back_to_start])) # Voltar ao Começo
def reset_manual_form():
    """Reseta os campos do formulário manual para valores padrão."""
    frm_nome.value = ''
    frm_cpf.value = ''
    frm_data_nasc.value = None
    frm_bairro_casa.value = ''
    frm_bairro_trabajo.value = ''
    frm_n_pessoas.value = 1
    frm_contaminado.value = False
    frm_familiar_cont.value = False
    frm_tempo_casa = widgets.IntText(description='Tempo em casa (dias)', value=0)
    frm_tempo_fora = widgets.IntText(description='Tempo fora casa (dias)', value=0)
    frm_salario.value = 0.0
    frm_bairro_trajeto.value = ''

# --------------------------

# WIDGETS: escolha do usuário e botões de ação

# --------------------------

choice = widgets.RadioButtons(
    options=["Subir arquivo (CSV/Excel)", "Adicionar manualmente"],
    description="Opção:",
    value="Adicionar manualmente" # Changed default value to "Adicionar manualmente"
)

btn_upload = widgets.Button(description=" Subir arquivo", button_style='primary')
btn_upload.on_click(on_click_upload)

btn_add_more = widgets.Button(description=" Preencher novo entrevistado", button_style='info')
btn_add_more.on_click(on_click_add_more)

btn_finalize = widgets.Button(description=" Finalizar", button_style='success')
btn_finalize.on_click(on_click_finalize)

btn_view_all = widgets.Button(description=" Ver todos", button_style='info')
btn_view_all.on_click(on_click_view_all)

btn_back_to_start = widgets.Button(description=" Voltar ao início", button_style='warning')
btn_back_to_start.on_click(on_click_back_to_start)

btn_delete_all = widgets.Button(description=" Deletar toda a base de dados", button_style='danger') # New delete button
btn_delete_all.on_click(on_click_delete_all) # Assign click handler


# ---------- Formulário manual ----------

frm_nome = widgets.Text(description='Nome', placeholder='Nome completo')
frm_cpf = widgets.Text(description='CPF', placeholder='Somente números (11 dígitos)')
frm_data_nasc = widgets.DatePicker(description='Data Nasc')

# Usando Dropdown para os campos de Bairro

frm_bairro_casa = widgets.Dropdown(options=[''] + UBATUBA_BAIRROS, description='Bairro Casa')
frm_bairro_trabajo = widgets.Dropdown(options=[''] + UBATUBA_BAIRROS, description='Bairro Trabalho')
frm_bairro_trajeto = widgets.Dropdown(options=[''] + UBATUBA_BAIRROS, description='Bairro Intermediario') # Novo nome

frm_n_pessoas = widgets.IntText(description='N Pessoas Casa', value=1)
frm_contaminado = widgets.Checkbox(description='Contaminado?')
frm_familiar_cont = widgets.Checkbox(description='Familiar contaminado?')
frm_tempo_casa = widgets.IntText(description='Tempo em casa (dias)', value=0)
frm_tempo_fora = widgets.IntText(description='Tempo fora casa (dias)', value=0)
frm_salario = widgets.FloatText(description='Salário mensal', value=0.0)

btn_submit_manual = widgets.Button(description=" Enviar entrevistado", button_style='success')
btn_submit_manual.on_click(on_submit_manual)

# Atualizando o VBox do formulário manual

manual_form = widgets.VBox([
    frm_nome, frm_cpf, frm_data_nasc, frm_bairro_casa, frm_bairro_trabajo, # Usando novos campos de bairro
    frm_n_pessoas, frm_contaminado, frm_familiar_cont, frm_tempo_casa,
    frm_tempo_fora, frm_salario, frm_bairro_trajeto,  # Usando novo campo de bairro
    btn_submit_manual
])

# --------------------------

# UI Dinâmica: alterna entre upload e manual

# --------------------------

output_info = widgets.HTML(
    "<b>Escolha 'Subir arquivo' para importar arquivo CSV/XLSX ou 'Adicionar manualmente' para preencher um formulário com os dados.</b>"
)

# Create a dedicated output widget for the dynamic content

output_area = widgets.Output()


def on_choice_change(change):
    with output_area:
        clear_output(wait=True)  # Clear previous content in the output area
        selected_option = change['new']

        if selected_option == "Subir arquivo (CSV/Excel)":
            display(widgets.HTML("<h4>Passo a passo — Subir arquivo</h4>"
                         "<ol>"
                         "<li>Clique em <b> Subir arquivo</b>.</li>"
                         "<li>Selecione um arquivo .csv ou .xlsx (colunas com nomes compatíveis).</li>"
                         "<li>O sistema processará o arquivo e adicionará os registros à base automaticamente.</li>"
                         "</ol>"
                         "<b>Dica:</b> A coluna deve ter cabeçalhos que correspondam às informações "
                         "(Exemplos: CPF, Nome, Data_Nascimento, Bairro Casa, Bairro Trabalho, N_Pessoas_Casa, Contaminado, Familiar_Contaminado, Tempo_Em_Casa, Tempo_Fora_Casa, Salario, Bairro Intermediario, Data_Envio, Amostra, Bairro). As colunas antigas de CEP, Latitude e Longitude serão ignoradas ou removidas se presentes no arquivo."))
            display(btn_upload)
            display(btn_delete_all) # Add delete button
        elif selected_option == "Adicionar manualmente":
            display(widgets.HTML("<h4>Formulário manual</h4><b>Preencha os campos e clique em 'Enviar entrevistado'.</b>"))
            display(manual_form) # Explicitly display the manual_form VBox within the output area
            reset_manual_form() # Reset form fields
            display(btn_delete_all) # Add delete button
        # Removed the logic that changed choice options and handled temporary states


# Observe the choice changes

choice.observe(on_choice_change, names='value')

# --------------------------

# MONTAGEM FINAL DA UI(Interface do Usuário)

# --------------------------

ui_box = widgets.VBox([
    widgets.HTML("<h2> Aquisição de Dados — Parte 1</h2>"),
    choice,
    output_info,
    output_area # Include the dedicated output area for dynamic content
])

# Mostrar interface inicial

display(ui_box)

# Acione a exibição inicial com base na opção padrão.

on_choice_change({'new': choice.value})

# Exibição inicial da base abaixo da interface do usuário (opcional, pois será exibida após as ações).
# Proximo passo: decidir como lidar com os dados ausentes, converter tipos, etc.
'''

In [ ]:
'''
#--------------------------------------------------------------------------------------------------

# 1. Aquisição de Dados (Parte 2)
#    - Excluir dados

#--------------------------------------------------------------------------------------------------

# Código de respaldo para corrigir, caso algum dado tenha broke a base de dados.

import ipywidgets as widgets
from IPython.display import display, clear_output
# Crie uma lista suspensa(dropdown) para selecionar o entrevistado à ser exclído.
# Utilizando o CPF, pois é o identificador único.

cpf_to_delete_widget = widgets.Dropdown(
    options=[''] + base_df['CPF'].astype(str).tolist(),
    description='Selecionar CPF:',
    disabled=False,
)

# Crie um botão para acionar a exclusão.
btn_delete_interviewee = widgets.Button(
    description='Excluir Entrevistado',
    button_style='danger', # Cor vermelha para ações  destrutivas
    disabled=False,
)

# Área de saída para exibir mensagens
delete_output_area = widgets.Output()

def on_delete_button_click(b):
    """Handles the click event for the delete button."""
    with delete_output_area:
        clear_output(wait=True)
        selected_cpf = cpf_to_delete_widget.value

        if not selected_cpf:
            display(widgets.HTML("<b style='color:red'>Por favor, selecione um CPF para excluir.</b>"))
            return

        global base_df
        initial_rows = len(base_df)

        # Encontre o índice da linha com o CPF selecionado.
        # Use .index para obter os rótulos de índice reais, o que é mais seguro para descarte.
        index_to_delete = base_df[base_df['CPF'].astype(str) == selected_cpf].index

        if not index_to_delete.empty:
            # Drop the row(s) with the found index
            base_df = base_df.drop(index_to_delete)
            display(widgets.HTML(f"<b>Entrevistado com CPF {selected_cpf} excluído com sucesso.</b>"))
            # Update the dropdown options after deletion
            cpf_to_delete_widget.options = [''] + base_df['CPF'].astype(str).tolist()
        else:
            display(widgets.HTML(f"<b style='color:orange'>Nenhum entrevistado encontrado com CPF {selected_cpf}.</b>"))

        # Opcionalmente, mostre o cabeçalho do dataframe atualizado.
        display(base_df.head())

# Vincula o clique do botão à função de tratamento.
btn_delete_interviewee.on_click(on_delete_button_click)

# Display the widgets
display(widgets.VBox([
    cpf_to_delete_widget,
    btn_delete_interviewee,
    delete_output_area
]))
'''

In [ ]:
'''
#--------------------------------------------------------------------------------------------------

# 1 . Aquisição de Dados (Parte 3)
#   - Confirmação de dados à Processar

#--------------------------------------------------------------------------------------------------

# Código de Respaudo para saber quais dados foram colocados, caso haja erros nas células acima.

mostrar_base(df_to_show=base_df, title="Base de dados de todos entrevistados")
'''

In [ ]:
# Este Código é apenas de respaldo para troca de coordenadas mais rápido

'''
import re

def dms_para_decimal(grau, minuto, segundo, direcao):
    """Converte coordenadas em graus, minutos e segundos (DMS) para decimal."""
    decimal = float(grau) + float(minuto)/60 + float(segundo)/3600
    if direcao in ['S', 'W']:
        decimal *= -1
    return decimal

def converter_coordenadas_google_earth(texto):
    """
    Converte texto no formato do Google Earth (ex: 23°43'12.5"S 45°02'45.2"W)
    em coordenadas decimais.
    """
    padrao = r'(\d+)°(\d+)\'([\d\.]+)"?([NS])\s+(\d+)°(\d+)\'([\d\.]+)"?([EW])'
    match = re.search(padrao, texto.strip())

    if not match:
        raise ValueError("Formato inválido! Exemplo válido: 23°43'12.5\"S 45°02'45.2\"W")

    lat_g, lat_m, lat_s, lat_dir, lon_g, lon_m, lon_s, lon_dir = match.groups()
    lat_decimal = dms_para_decimal(lat_g, lat_m, lat_s, lat_dir)
    lon_decimal = dms_para_decimal(lon_g, lon_m, lon_s, lon_dir)

    return lat_decimal, lon_decimal

def obter_coordenadas_bairros():
    bairros = [
        "Centro", "Itaguá", "Praia Grande", "Toninhas", "Enseada",
        "Lazaro", "Domenico", "Estufa II", "Rio Escuro", "Perequê-Açu",
        "Sertão da Quina", "Maranduba", "Sapê", "Tabatinga", "Cunhambebe",
        "Mata Atlântica", "Rio da Prata", "Folha Seca", "Figueira", "Sesmaria",
        "Ipiranguinha", "Taquaral", "Bela Vista", "Jardim Carolina",
        "Ressaca", "Pedreira", "Semidouro", "Horto", "Monte Valério"
    ]

    bairros_coords = {}

    print("Digite as coordenadas de cada bairro no formato do Google Earth:")
    print("Exemplo: 23°43'12.5\"S 45°02'45.2\"W\n")

    for bairro in bairros:
        while True:
            try:
                coords = input(f"{bairro}: ").strip()
                lat, lon = converter_coordenadas_google_earth(coords)
                bairros_coords[bairro] = {"latitude": lat, "longitude": lon}
                break
            except ValueError as e:
                print(e)
                print("Tente novamente.\n")

    print("\nDicionário final:\n")
    print("bairros_coords = {")
    for b, c in bairros_coords.items():
        print(f'    "{b}": {{"latitude": {c["latitude"]:.6f}, "longitude": {c["longitude"]:.6f}}},')
    print("}")

    return bairros_coords

# Executar o programa
if __name__ == "__main__":
    bairros_coords = obter_coordenadas_bairros()


'''


In [ ]:
# ...........Passos..................
# 1. Aquisição de Dados
# 2. Processamento de Dados
# 3. Análise e Manipulação de Dados
# 4. Exibição dos Dados
#....................................
# -------------------------------------------------------------------------------------------------
# 1. Aquisição de Dados (Parte 1)
# -------------------------------------------------------------------------------------------------

#-------------!!Código feito com base no Google.Colab resources!!---------------------------------

# Código para ler arquivo CSV ou Excel e acrescentar a base de dados cada pessoa.
# Importando Biblioteca e Recursos.

import pandas as pd
from datetime import datetime
from google.colab import files
import io
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import folium
from folium.plugins import HeatMap

# ----------------------

# Colunas esperadas
EXPECTED_COLS = [
    "CPF", "Nome", "Data_Nascimento", "Bairro Casa", "Bairro Trabalho",
    "N_Pessoas_Casa", "Contaminado", "Familiar_Contaminado",
    "Tempo_Em_Casa", "Tempo_Fora_Casa", "Salario",
    "Bairro Intermediario", "Data_Envio"
]

# -----------------------
# Lista de Bairros : Opções
# -----------------------
UBATUBA_BAIRROS = [
    "Centro", "Itaguá", "Praia Grande", "Toninhas", "Enseada",
    "Lazaro", "Domenico", "Estufa II", "Rio Escuro", "Perequê-Açu",
    "Sertão da Quina", "Maranduba", "Sapê", "Tabatinga", "Cunhambebe",
    "Mata Atlântica", "Rio da Prata", "Folha Seca", "Figueira", "Sesmaria",
    "Ipiranguinha", "Taquaral", "Bela Vista", "Jardim Carolina",
    "Ressaca", "Pedreira", "Semidouro", "Horto", "Monte Valério"
]
UBATUBA_BAIRROS.sort()

# Dicionário associando bairros a latitudes e longitudes
bairros_coords = {
    "Centro": {"latitude": -23.435556, "longitude": -45.073889},
    "Itaguá": {"latitude": -23.455000, "longitude": -45.066667},
    "Praia Grande": {"latitude": -23.467222, "longitude": -45.065000},
    "Toninhas": {"latitude": -23.487222, "longitude": -45.075833},
    "Enseada": {"latitude": -23.491389, "longitude": -45.091111},
    "Lazaro": {"latitude": -23.501944, "longitude": -45.132500},
    "Domenico": {"latitude": -23.497500, "longitude": -45.141667},
    "Estufa II": {"latitude": -23.455556, "longitude": -45.080833},
    "Rio Escuro": {"latitude": -23.487778, "longitude": -45.154167},
    "Perequê-Açu": {"latitude": -23.414444, "longitude": -45.063333},
    "Sertão da Quina": {"latitude": -23.530000, "longitude": -45.243333},
    "Maranduba": {"latitude": -23.538889, "longitude": -45.234722},
    "Sapê": {"latitude": -23.528056, "longitude": -45.229167},
    "Tabatinga": {"latitude": -23.569722, "longitude": -45.268611},
    "Cunhambebe": {"latitude": -22.961389, "longitude": -44.440000},
    "Mata Atlântica": {"latitude": -23.456944, "longitude": -45.117500},
    "Rio da Prata": {"latitude": -23.556389, "longitude": -45.248333},
    "Folha Seca": {"latitude": -23.482222, "longitude": -45.166111},
    "Figueira": {"latitude": -23.396111, "longitude": -45.123889},
    "Sesmaria": {"latitude": -23.467500, "longitude": -45.090000},
    "Ipiranguinha": {"latitude": -23.428333, "longitude": -45.110833},
    "Taquaral": {"latitude": -23.406111, "longitude": -45.065000},
    "Bela Vista": {"latitude": -23.442500, "longitude": -45.100278},
    "Jardim Carolina": {"latitude": -23.437778, "longitude": -45.090833},
    "Ressaca": {"latitude": -23.429722, "longitude": -45.089722},
    "Pedreira": {"latitude": -23.419722, "longitude": -45.072222},
    "Semidouro": {"latitude": -23.408889, "longitude": -45.066111},
    "Horto": {"latitude": -23.421111, "longitude": -45.108056},
    "Monte Valério": {"latitude": -23.452500, "longitude": -45.106944},
}

zoom_level=13
bairros_coords_df = pd.DataFrame.from_dict(bairros_coords, orient='index')
bairros_coords_df.index.name = 'Bairro'

# --------------------------
# BASE GLOBAL (DataFrame)
# --------------------------
if 'base_df' not in globals():
    base_df = pd.DataFrame(columns=EXPECTED_COLS)

if 'session_manual_df' not in globals():
    session_manual_df = pd.DataFrame(columns=EXPECTED_COLS)

# --------------------------
# FUNÇÕES AUXILIARES
# --------------------------
def ensure_columns(df):
    for c in EXPECTED_COLS:
        if c not in df.columns:
            df[c] = pd.NA
    return df[EXPECTED_COLS]

def add_from_dataframe(df_new):
    global base_df
    base_df = pd.concat([base_df, ensure_columns(df_new)], ignore_index=True)
    with output_area:
        clear_output(wait=True)
        display(widgets.HTML("<b>Arquivo adicionado à base de dados com sucesso.</b>"))
        display(mostrar_base())
        display(widgets.VBox([btn_view_all, btn_back_to_start, btn_delete_all]))

def parse_uploaded_filecontent(filename, content_bytes):
    try:
        if filename.lower().endswith('.csv'):
            return pd.read_csv(io.BytesIO(content_bytes))
        elif filename.lower().endswith(('.xls', '.xlsx')):
            return pd.read_excel(io.BytesIO(content_bytes))
        else:
            raise ValueError("Formato de arquivo sem suporte. Use os formatos .csv ou .xlsx/.xls")
    except Exception as e:
        raise

def mostrar_base(df_to_show=None, title="Base de entrevistados (atual)"):
    df_display = df_to_show if df_to_show is not None else base_df
    if len(df_display) == 0:
        return widgets.HTML("<b> Base vazia.</b>")
    return widgets.HTML(df_display.astype(object).fillna("").infer_objects(copy=False).to_html())

# --------------------------------------------------
# Funções para Manipulações: Ações da interface do usuário
# --------------------------------------------------

#Função para a versão de Computador(APP\via API), para Python foi criada outras funcções devido á necessidade das céluls restarts
#def on_click_upload(_):
    #with output_area:
        #clear_output(wait=True)
        #display(widgets.HTML("<b>⬆️ Selecione o arquivo CSV ou Excel para importar.</b>"))

        # Upload deve ocorrer diretamente no contexto da célula:
        #print("Aguarde: Abrindo seletor de arquivo...")
        #uploaded = files.upload()  # <-- Interface aparecerá corretamente aqui

        #if not uploaded:
            #display(widgets.HTML("<b style='color:red'>Nenhum arquivo enviado.</b>"))
            #display(ui_box)
            #return

        #for filename, fileinfo in uploaded.items():
            #try:
                #df_new = parse_uploaded_filecontent(filename, fileinfo['content'])
            #except Exception as e:
                #display(widgets.HTML(f"<b style='color:red'>Erro ao ler {filename}: {e}</b>"))
                #continue

            # Conversão automática de colunas com datas
            #for col in df_new.columns:
                #if any(x in col.lower() for x in ['data', 'envio', 'nascimento']):
                    #try:
                        #df_new[col] = pd.to_datetime(df_new[col]).dt.date
                    #except Exception:
                        #pass

            #add_from_dataframe(df_new)

#Função acima, na versão de Python para (Colab)
def on_click_upload(_):
    with output_area:
        clear_output(wait=True)
        display(widgets.HTML("""
        <b>🚀 Etapas para enviar seu arquivo:</b>
        <ol>
          <li>Execute a célula abaixo para abrir o seletor de arquivo.</li>
          <li>Escolha o arquivo CSV ou Excel no seu computador.</li>
        </ol>
        """))
        display(HTML("<b>➡️ Execute a próxima célula para fazer o upload:</b>"))


def on_submit_manual(_):
    global session_manual_df
    cpf_value = frm_cpf.value.strip()
    with output_area:
        clear_output(wait=True)
        if not cpf_value.isdigit() or len(cpf_value) != 11:
             display(widgets.HTML("<b style='color:red'>Erro: CPF deve conter 11 dígitos numéricos.</b>"))
             display(manual_form)
             display(btn_delete_all)
             return
        dados = {
            "CPF": cpf_value,
            "Nome": frm_nome.value.strip(),
            "Data_Nascimento": frm_data_nasc.value.strftime("%Y-%m-%d") if frm_data_nasc.value else pd.NA,
            "Bairro Casa": frm_bairro_casa.value,
            "Bairro Trabalho": frm_bairro_trabajo.value,
            "N_Pessoas_Casa": frm_n_pessoas.value,
            "Contaminado": frm_contaminado.value,
            "Familiar_Contaminado": frm_familiar_cont.value,
            "Tempo_Em_Casa": frm_tempo_casa.value,
            "Tempo_Fora_Casa": frm_tempo_fora.value,
            "Salario": frm_salario.value,
            "Bairro Intermediario": frm_bairro_trajeto.value,
            "Data_Envio": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        df_temp = ensure_columns(pd.DataFrame([dados]))
        session_manual_df = pd.concat([session_manual_df, df_temp], ignore_index=True)
        display(widgets.VBox([
            widgets.HTML("<b>Entrevistado adicionado à base temporária da sessão.</b>"),
            mostrar_base(df_to_show=session_manual_df, title="Sessão Atual"),
            widgets.VBox([btn_add_more, btn_finalize, btn_view_all, btn_back_to_start, btn_delete_all])
        ]))

def on_click_add_more(_):
    with output_area:
        clear_output(wait=True)
        display(widgets.HTML("<h4>Formulário manual</h4><b>Preencha os campos e clique em 'Enviar entrevistado'.</b>"))
        display(manual_form)
        reset_manual_form()
        display(btn_delete_all)

def on_click_finalize(_):
    global base_df, session_manual_df
    rows_added_this_session = len(session_manual_df)
    if rows_added_this_session > 0:
        base_df = pd.concat([base_df, session_manual_df], ignore_index=True)
        with output_area:
            clear_output(wait=True)
            display(widgets.HTML("<b> Adição manual finalizada.</b>"))
            display(mostrar_base(df_to_show=base_df.tail(rows_added_this_session)))
    else:
        with output_area:
            clear_output(wait=True)
            display(widgets.HTML("<b> Nenhum entrevistado adicionado nesta sessão.</b>"))
    session_manual_df = pd.DataFrame(columns=EXPECTED_COLS)
    display(widgets.VBox([btn_view_all, btn_back_to_start, btn_delete_all]))

def on_click_view_all(_):
    with output_area:
        clear_output(wait=True)
        display(mostrar_base(df_to_show=base_df, title="Base de dados completa"))
        display(widgets.VBox([btn_back_to_start, btn_delete_all]))

def on_click_back_to_start(_):
    with output_area:
        clear_output(wait=True)
        display(output_info)
        display(choice)
        on_choice_change({'new': choice.value})

def on_click_delete_all(_):
    global base_df
    base_df = pd.DataFrame(columns=EXPECTED_COLS)
    with output_area:
        clear_output(wait=True)
        display(widgets.HTML("<b>Base de dados deletada.</b>"))
        display(mostrar_base())
        display(widgets.VBox([btn_back_to_start]))

def reset_manual_form():
    frm_nome.value = ''
    frm_cpf.value = ''
    frm_data_nasc.value = None
    frm_bairro_casa.value = ''
    frm_bairro_trabajo.value = ''
    frm_n_pessoas.value = 1
    frm_contaminado.value = False
    frm_familiar_cont.value = False
    frm_tempo_casa.value = 0
    frm_tempo_fora.value = 0
    frm_salario.value = 0.0
    frm_bairro_trajeto.value = ''

# --------------------------
# WIDGETS E INTERFACE PRINCIPAL
# --------------------------
choice = widgets.RadioButtons(
    options=["Subir arquivo (CSV/Excel)", "Adicionar manualmente"],
    description="Opção:",
    value="Subir arquivo (CSV/Excel)"
)

btn_upload = widgets.Button(description=" Subir arquivo", button_style='primary')
btn_upload.on_click(on_click_upload)

btn_add_more = widgets.Button(description=" Preencher novo entrevistado", button_style='info')
btn_add_more.on_click(on_click_add_more)

btn_finalize = widgets.Button(description=" Finalizar", button_style='success')
btn_finalize.on_click(on_click_finalize)

btn_view_all = widgets.Button(description=" Ver todos", button_style='info')
btn_view_all.on_click(on_click_view_all)

btn_back_to_start = widgets.Button(description=" Voltar ao início", button_style='warning')
btn_back_to_start.on_click(on_click_back_to_start)

btn_delete_all = widgets.Button(description=" Deletar toda a base de dados", button_style='danger')
btn_delete_all.on_click(on_click_delete_all)

frm_nome = widgets.Text(description='Nome', placeholder='Nome completo')
frm_cpf = widgets.Text(description='CPF', placeholder='Somente números (11 dígitos)')
frm_data_nasc = widgets.DatePicker(description='Data Nasc')
frm_bairro_casa = widgets.Dropdown(options=[''] + UBATUBA_BAIRROS, description='Bairro Casa')
frm_bairro_trabajo = widgets.Dropdown(options=[''] + UBATUBA_BAIRROS, description='Bairro Trabalho')
frm_bairro_trajeto = widgets.Dropdown(options=[''] + UBATUBA_BAIRROS, description='Bairro Intermediario')
frm_n_pessoas = widgets.IntText(description='N Pessoas Casa', value=1)
frm_contaminado = widgets.Checkbox(description='Contaminado?')
frm_familiar_cont = widgets.Checkbox(description='Familiar contaminado?')
frm_tempo_casa = widgets.IntText(description='Tempo em casa (dias)', value=0)
frm_tempo_fora = widgets.IntText(description='Tempo fora casa (dias)', value=0)
frm_salario = widgets.FloatText(description='Salário mensal', value=0.0)

btn_submit_manual = widgets.Button(description=" Enviar entrevistado", button_style='success')
btn_submit_manual.on_click(on_submit_manual)

manual_form = widgets.VBox([
    frm_nome, frm_cpf, frm_data_nasc, frm_bairro_casa, frm_bairro_trabajo,
    frm_n_pessoas, frm_contaminado, frm_familiar_cont, frm_tempo_casa,
    frm_tempo_fora, frm_salario, frm_bairro_trajeto, btn_submit_manual
])

output_info = widgets.HTML(
    "<b>Escolha 'Subir arquivo' para importar arquivo CSV/XLSX ou 'Adicionar manualmente' para preencher um formulário com os dados.</b>"
)
output_area = widgets.Output()

def on_choice_change(change):
    with output_area:
        clear_output(wait=True)
        selected_option = change['new']
        if selected_option == "Subir arquivo (CSV/Excel)":
            display(widgets.HTML("<h4>Passo a passo — Subir arquivo</h4>"
                         "<ol><li>Clique em <b> Subir arquivo</b>.</li>"
                         "<li>Selecione um arquivo .csv ou .xlsx.</li></ol>"))
            display(btn_upload)
            display(btn_delete_all)
        elif selected_option == "Adicionar manualmente":
            display(widgets.HTML("<h4>Formulário manual</h4><b>Preencha os campos e clique em 'Enviar entrevistado'.</b>"))
            display(manual_form)
            reset_manual_form()
            display(btn_delete_all)

choice.observe(on_choice_change, names='value')

ui_box = widgets.VBox([
    widgets.HTML("<h2> Aquisição de Dados — Parte 1</h2>"),
    choice,
    output_info,
    output_area
])

display(ui_box)
on_choice_change({'new': choice.value})

# --------------------------------------------------------------------------------------------------
# 1. Aquisição de Dados (Parte 2) - Excluir dados
# --------------------------------------------------------------------------------------------------
cpf_to_delete_widget = widgets.Dropdown(
    options=[''] + base_df['CPF'].astype(str).tolist(),
    description='Selecionar CPF:',
)
btn_delete_interviewee = widgets.Button(
    description='Excluir Entrevistado',
    button_style='danger',
)
delete_output_area = widgets.Output()

def on_delete_button_click(b):
    with delete_output_area:
        clear_output(wait=True)
        selected_cpf = cpf_to_delete_widget.value
        if not selected_cpf:
            display(widgets.HTML("<b style='color:red'>Por favor, selecione um CPF para excluir.</b>"))
            return
        global base_df
        index_to_delete = base_df[base_df['CPF'].astype(str) == selected_cpf].index
        if not index_to_delete.empty:
            base_df = base_df.drop(index_to_delete)
            display(widgets.HTML(f"<b>Entrevistado com CPF {selected_cpf} excluído com sucesso.</b>"))
            cpf_to_delete_widget.options = [''] + base_df['CPF'].astype(str).tolist()
        else:
            display(widgets.HTML(f"<b style='color:orange'>Nenhum entrevistado encontrado com CPF {selected_cpf}.</b>"))
        display(base_df.head())

btn_delete_interviewee.on_click(on_delete_button_click)

display(widgets.VBox([
    widgets.HTML("<h3>Excluir Entrevistado</h3>"),
    cpf_to_delete_widget,
    btn_delete_interviewee,
    delete_output_area
]))

# --------------------------------------------------------------------------------------------------
# 1 . Aquisição de Dados (Parte 3)
#   - Confirmação de dados à Processar
# --------------------------------------------------------------------------------------------------
#
display(mostrar_base(df_to_show=base_df))

In [ ]:
# Célula para fazer Upload
# 🚀 Upload direto (executar esta célula quando quiser subir arquivo)
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()

if uploaded:
    for filename, fileinfo in uploaded.items():
        print(f"📄 Arquivo recebido: {filename}")
        if filename.lower().endswith('.csv'):
            df_new = pd.read_csv(io.BytesIO(fileinfo))
        elif filename.lower().endswith(('.xls', '.xlsx')):
            df_new = pd.read_excel(io.BytesIO(fileinfo))
        else:
            raise ValueError("Formato de arquivo não suportado.")

        # Adiciona à base
        base_df = pd.concat([base_df, df_new], ignore_index=True)
        print(f"✅ {len(df_new)} registros adicionados à base.")


In [ ]:
#--------------------------------------------------------------------------------------------------

# 2. Análise e Manipulação de Dados (Parte 1)
#    - Densidade de bairros e agregar coordenadas

#--------------------------------------------------------------------------------------------------

import pandas as pd
from datetime import datetime
from google.colab import files
import io
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import folium
from folium.plugins import HeatMap

# 1. Aggregate data by neighborhood
# Use 'Bairro Casa' for population density.
densidade_bairros_df = base_df.groupby('Bairro Casa')['N_Pessoas_Casa'].sum().reset_index()
densidade_bairros_df = densidade_bairros_df.rename(columns={'N_Pessoas_Casa': 'Total_Pessoas'})

# Ensure Neighborhood names are strings and stripped for accurate merging
densidade_bairros_df['Bairro Casa'] = densidade_bairros_df['Bairro Casa'].astype(str).str.strip()
bairros_coords_df.index = bairros_coords_df.index.astype(str).str.strip()


# 2. Combine with coordinates
# Ensure that the index of bairros_coords_df is the neighborhood name.
if bairros_coords_df.index.name != 'Bairro':
     bairros_coords_df = bairros_coords_df.copy()
     bairros_coords_df.index.name = 'Bairro'

heatmap_data = pd.merge(densidade_bairros_df, bairros_coords_df, left_on='Bairro Casa', right_index=True, how='left')

# Report neighborhoods that were not found a match.
missing_coords = heatmap_data[heatmap_data['latitude'].isna()]['Bairro Casa'].tolist()
if missing_coords:
    print(f"Aviso: Não foram encontradas coordenadas para os seguintes bairros: {', '.join(missing_coords)}")

# Checar e converter os tipos de dados se necessário
heatmap_data['Total_Pessoas'] = pd.to_numeric(heatmap_data['Total_Pessoas'], errors='coerce')

# Identify coordinate and weight columns
latitude_col = 'latitude'
longitude_col = 'longitude'
weight_col = 'Total_Pessoas'

# Drop rows with missing values in the necessary columns
heatmap_data_cleaned = heatmap_data.dropna(subset=[latitude_col, longitude_col, weight_col]).copy()

# Create the list of data points for the heatmap
heatmap_points = heatmap_data_cleaned[[latitude_col, longitude_col, weight_col]].values.tolist()

if not heatmap_points:
    print("Sem dados válidos com coordenadas para gerar o heatmap.")
    # Optionally, return or set heatmap_points to None if you want to handle this case later
    heatmap_points = []

In [ ]:
# -------------------------------------------------------------------------------------------------
# 3. Análise e Manipulação de Dados
# -------------------------------------------------------------------------------------------------

# Verificar os tipos de dados das colunas
# display(base_df.info())

# Converter colunas numéricas se necessário (já tratado na aquisição, mas bom verificar)
for col in ['N_Pessoas_Casa', 'Tempo_Em_Casa', 'Tempo_Fora_Casa', 'Salario']:
    if col in base_df.columns:
        # Convert to numeric, coercing errors to NaN
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce')
        # Fill NaN values with a suitable default (e.g., 0) if appropriate for analysis
        base_df[col] = base_df[col].fillna(0)

# Converter colunas booleanas se necessário (já tratado na aquisição, mas bom verificar)
for col in ['Contaminado', 'Familiar_Contaminado']:
    if col in base_df.columns:
        # Convert to boolean, coercing errors to False for common string representations
        # If the data is already boolean from manual entry, this won't change it.
        # If it's string like 'Sim'/'Não' or 'True'/'False', convert. Assuming 'Sim'/'True' means True.
        if pd.api.types.is_object_dtype(base_df[col].dtype):
            base_df[col] = base_df[col].apply(lambda x: True if str(x).lower() in ['sim', 'true'] else False)
        # If it's numeric (e.g., 1/0), convert
        elif pd.api.types.is_numeric_dtype(base_df[col].dtype):
             base_df[col] = base_df[col].astype(bool)


# Calcular a porcentagem de infectados por bairro
# Certificar-se de que 'Bairro Casa' e 'N_Pessoas_Casa' existem e têm tipos corretos
if 'Bairro Casa' in base_df.columns and 'N_Pessoas_Casa' in base_df.columns and 'Contaminado' in base_df.columns:
    # Calculate the total number of people and infected people per neighborhood
    bairro_infected_summary = base_df.groupby('Bairro Casa').agg(
        Total_Pessoas=('N_Pessoas_Casa', 'sum'),
        Total_Contaminados=('Contaminado', lambda x: x.sum() if pd.api.types.is_bool_dtype(x) else (x == True).sum()) # Sum boolean True values
    ).reset_index()

    # Calculate the percentage of infected individuals
    bairro_infected_summary['Porcentagem_Infectados'] = (bairro_infected_summary['Total_Contaminados'] / bairro_infected_summary['Total_Pessoas']) * 100

    # Handle potential division by zero if a neighborhood has 0 total people
    bairro_infected_summary['Porcentagem_Infectados'] = bairro_infected_summary['Porcentagem_Infectados'].fillna(0)

    # Rename for clarity and assign to bairro_summary_df
    bairro_summary_df = bairro_infected_summary.rename(columns={'Bairro Casa': 'Bairro'})

    # Exibir o DataFrame de resumo por bairro (opcional, para verificação)
    # display(bairro_summary_df)


# Definir função para atribuir recomendação com base na porcentagem de infectados
def assign_recommendation(percentage):
    if percentage >= 10:
        return "Região de risco"
    elif percentage >= 5:
        return "Região de atenção"
    else:
        return "Região saudável"

# Apply the function to create a new column with recommendations
bairro_summary_df['Recomendacao'] = bairro_summary_df['Porcentagem_Infectados'].apply(assign_recommendation)

# Display the updated DataFrame
# display(bairro_summary_df)

In [ ]:
# -------------------------------------------------------------------------------------------------
# 4. Exibição dos Dados
# -------------------------------------------------------------------------------------------------

# display(bairro_summary_df)

# Preparar dados para o mapa de calor (usando o DataFrame de resumo por bairro)
# Garantir que o merge usa a coluna 'Bairro' do summary_df e o index de bairros_coords_df
heatmap_data = pd.merge(bairro_summary_df, bairros_coords_df, left_on='Bairro', right_index=True, how='left')

# Check for missing coordinates after merge
missing_coords_heatmap = heatmap_data[heatmap_data['latitude'].isna()]['Bairro'].tolist()
if missing_coords_heatmap:
    print(f"Aviso: Não foram encontradas coordenadas para os seguintes bairros (para o mapa de calor): {', '.join(missing_coords_heatmap)}")

# Remove rows with missing coordinates for heatmap to prevent errors
heatmap_data_cleaned = heatmap_data.dropna(subset=['latitude', 'longitude']).copy()


# Criar lista de pontos para o heatmap [latitude, longitude, peso]
# Usar 'Total_Pessoas' ou 'Porcentagem_Infectados' como peso.
# Se usar 'Total_Pessoas': heatmap_points = heatmap_data_cleaned[['latitude', 'longitude', 'Total_Pessoas']].values.tolist()
# Se usar 'Porcentagem_Infectados':
heatmap_points = heatmap_data_cleaned[['latitude', 'longitude', 'Porcentagem_Infectados']].values.tolist()


# Criar o mapa de calor
# Approximate central coordinates for Ubatuba, or use the mean of available coordinates
if not heatmap_data_cleaned.empty:
    map_center = [heatmap_data_cleaned['latitude'].mean(), heatmap_data_cleaned['longitude'].mean()]
else:
    map_center = [-23.4500, -45.0900] # Default Ubatuba coordinates

zoom_level = 13 # Adjust zoom level as needed
m = folium.Map(location=map_center, zoom_start=zoom_level)

# Adicionar a camada de heatmap ao mapa
if heatmap_points: # Only add heatmap if there are valid points
    HeatMap(heatmap_points, radius=15).add_to(m) # Adjust radius as needed

# Exibir o mapa de calor
print("Mapa de Calor da Porcentagem de Infectados por Bairro:")
display(m)

# -------------------------------------------------------------------------------------------------
# Análise e Visualização Adicional (Opcional)
# -------------------------------------------------------------------------------------------------

# Exemplo: Criar um gráfico de barras das recomendações por bairro
# import matplotlib.pyplot as plt
# import seaborn as sns

# if not bairro_summary_df.empty:
#     plt.figure(figsize=(10, 6))
#     sns.barplot(x='Bairro', y='Porcentagem_Infectados', hue='Recomendacao', data=bairro_summary_df, dodge=False)
#     plt.xticks(rotation=90)
#     plt.title('Porcentagem de Infectados e Recomendação por Bairro')
#     plt.ylabel('Porcentagem de Infectados (%)')
#     plt.tight_layout()
#     plt.show()
# else:
#     print("Não há dados de resumo por bairro para gerar o gráfico.")


# Exemplo: Criar um dashboard simples com ipywidgets
# dashboard_output = widgets.Output()

# with dashboard_output:
#     display(widgets.HTML("<h2>Dashboard de Análise por Bairro</h2>"))
#     if not bairro_summary_df.empty:
#         display(bairro_summary_df)
#         # Optionally display the map again or other visualizations
#         # display(m)
#     else:
#         display(widgets.HTML("<b>Base de dados vazia. Adicione dados para ver o dashboard.</b>"))

# # Display the dashboard output
# display(dashboard_output)

In [ ]:
#--------------------------------------------------------------------------------------------------

# 4. Exibição dos Dados (Parte 1)
#    - Dados e informações

#--------------------------------------------------------------------------------------------------

# VERSÃO DE CÓDIGO NÚMERO 1.0

# Exibindo somente as necessárias e não dados intermediários
#display(pd.merge(densidade_bairros_df, bairros_coords_df, left_on='Bairro Casa', right_index=True, how='left'))
#display(densidade_bairros_df) # Mostra Densidade de Bairros em números
#print("Formatted data for heatmap (first 5 points):")
#display(heatmap_points[:5])
#display(risk_data_df.head())
#display(bairro_risk_agg_df.head())
#display(bairro_risk_coords_df.head())
#display(contaminated_counts)
#display(total_residents_per_bairro.head())
#display(merged_df)
#display(low_contamination_bairros)
#display(bairro_summary_df)
#display(base_df[['Nome', 'Data_Nascimento', 'Idade']].head())
#display(bairro_salario_medio.head())
#display(bairro_idade_media.head())
#print("Formatted data for new risk heatmap (first 5 points):")
#display(heatmap_points_new_risk[:5])
#display(bairro_aggregated_data)

# ============================================================
# DASHBOARD COMPLETO - VISUALIZAÇÃO E ANÁLISE INTERATIVA
# ============================================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from datetime import datetime

# ============================================================
# Create bairro_risk_coords_df
# Merge bairro_summary_df and bairros_coords_df to get coordinates for each neighborhood
bairro_risk_coords_df = pd.merge(bairro_summary_df, bairros_coords_df, left_on='Bairro', right_index=True, how='left')

# Add a dummy 'Indice_Risco' column for visualization since the original code expected it
# In a real scenario, this would be calculated based on your risk criteria
bairro_risk_coords_df['Indice_Risco'] = bairro_risk_coords_df['Porcentagem_Infectados']

# Check for missing coordinates after merge (optional but recommended)
missing_coords_risk = bairro_risk_coords_df[bairro_risk_coords_df['latitude'].isna()]['Bairro'].tolist()
if missing_coords_risk:
    print(f"Aviso: Não foram encontradas coordenadas para os seguintes bairros (para o dashboard): {', '.join(missing_coords_risk)}")

# Drop rows with missing coordinates to prevent errors in plotting
bairro_risk_coords_df_cleaned = bairro_risk_coords_df.dropna(subset=['latitude', 'longitude']).copy()

# ============================================================
# 1️ INDICADORES GERAIS (KPIs)
# ============================================================
total_entrevistados = len(base_df)
total_contaminados = base_df["Contaminado"].sum()
# Avoid division by zero if total_entrevistados is 0
perc_contaminados = (total_contaminados / total_entrevistados) * 100 if total_entrevistados > 0 else 0

media_salario = base_df["Salario"].mean()
# Calculate Age if Data_Nascimento is available and not all NaT
if "Data_Nascimento" in base_df.columns and not base_df['Data_Nascimento'].isnull().all():
    # Ensure 'Data_Nascimento' is datetime objects before calculating age
    try:
        base_df['Data_Nascimento'] = pd.to_datetime(base_df['Data_Nascimento'])
        base_df['Idade'] = (pd.to_datetime('now').year - base_df['Data_Nascimento'].dt.year)
        media_idade = base_df["Idade"].mean()
    except Exception as e:
        print(f"Could not calculate age: {e}")
        media_idade = None
else:
    media_idade = None


data_atual = datetime.now().strftime("%d/%m/%Y")

# ============================================================
# 2️ GRÁFICOS INTERATIVOS
# ============================================================

# Risco por bairro (scatter) - Use the cleaned DataFrame
if not bairro_risk_coords_df_cleaned.empty:
    fig_risco = px.scatter(
        bairro_risk_coords_df_cleaned,
        x="latitude", y="longitude",
        color="Indice_Risco",
        hover_name="Bairro", # Use 'Bairro' from the merged DataFrame
        color_continuous_scale="RdYlGn_r",
        title=" Mapa de Risco por Coordenadas",
        template="plotly_white"
    )
else:
    fig_risco = go.Figure().add_annotation(
        x=0.5, y=0.5,
        text="Sem dados válidos para o mapa de risco.",
        showarrow=False,
        xref="paper", yref="paper"
    )


# Salário médio por bairro (barras) - Ensure bairro_salario_medio is defined
if 'bairro_salario_medio' in globals() and not bairro_salario_medio.empty:
    fig_salario = px.bar(
        bairro_salario_medio,
        x='Bairro', # Use the column name 'Bairro'
        y='Salario_Medio', # Use the column name 'Salario_Medio'
        title=" Salário Médio por Bairro",
        labels={"Bairro": "Bairro", "Salario_Medio": "Salário Médio"},
        color='Salario_Medio',
        color_continuous_scale="Blues"
    )
    fig_salario.update_layout(xaxis_tickangle=-45)
else:
     fig_salario = go.Figure().add_annotation(
        x=0.5, y=0.5,
        text="Sem dados de salário por bairro.",
        showarrow=False,
        xref="paper", yref="paper"
    )


# Distribuição de idade
if "Idade" in base_df.columns and not base_df['Idade'].isnull().all():
    fig_idade = px.histogram(
        base_df,
        x="Idade",
        nbins=20,
        title=" Distribuição de Idades dos Entrevistados",
        color_discrete_sequence=["#007acc"]
    )
else:
    fig_idade = go.Figure().add_annotation(
        x=0.5, y=0.5,
        text="Sem dados de idade para exibir.",
        showarrow=False,
        xref="paper", yref="paper"
    )


# Heatmap de risco (mapbox) - Use the cleaned DataFrame
if not bairro_risk_coords_df_cleaned.empty:
    fig_heat = px.density_mapbox(
        bairro_risk_coords_df_cleaned,
        lat="latitude", lon="longitude", z="Indice_Risco",
        radius=20,
        center=dict(lat=map_center[0], lon=map_center[1]) if 'map_center' in globals() else dict(lat=-23.55, lon=-46.63), # Use map_center if available
        zoom=zoom_level if 'zoom_level' in globals() else 10, # Use zoom_level if available
        mapbox_style="carto-positron",
        title=" Heatmap de Risco por Localização"
    )
else:
    fig_heat = go.Figure().add_annotation(
        x=0.5, y=0.5,
        text="Sem dados válidos para o heatmap de risco.",
        showarrow=False,
        xref="paper", yref="paper"
    )


# ============================================================
# 3️ ESTILO HTML / CSS (layout limpo e moderno)
# ============================================================
estilo_html = """
<style>
.dashboard {
    font-family: 'Segoe UI', sans-serif;
    background-color: #f7f9fc;
    padding: 20px;
}
.header {
    text-align: center;
    margin-bottom: 30px;
}
.kpis {
    display: flex;
    justify-content: space-around;
    flex-wrap: wrap;
    margin-bottom: 30px;
}
.kpi {
    background: linear-gradient(135deg, #007acc, #00b4d8);
    color: white;
    padding: 25px;
    border-radius: 15px;
    text-align: center;
    width: 22%;
    min-width: 200px;
    box-shadow: 0 3px 10px rgba(0,0,0,0.2);
}
.kpi h2 {
    font-size: 2.2em;
    margin: 0;
}
.cards {
    display: flex;
    flex-wrap: wrap;
    justify-content: space-around;
    gap: 20px;
}
.card {
    background: white;
    border-radius: 12px;
    box-shadow: 0 3px 8px rgba(0,0,0,0.15);
    padding: 16px;
    width: 45%;
    min-width: 400px;
}
.card h3 {
    color: #003366;
    border-bottom: 2px solid #007acc;
    padding-bottom: 6px;
    margin-bottom: 12px;
}
.card table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}
.card th {
    background-color: #f0f4f8;
    color: #003366;
    text-align: left;
    padding: 6px;
}
.card td {
    padding: 6px;
    border-bottom: 1px solid #e0e0e0;
}
</style>
"""

# Format media_idade_texto safely
media_idade_texto = f"{media_idade:.1f}" if media_idade is not None else "N/A"

# ============================================================
# 4️ LAYOUT DO DASHBOARD (CORRIGIDO)
# ============================================================
# Ensure dataframes are not empty before converting to HTML
bairro_risk_table_html = bairro_risk_coords_df_cleaned.head(10).to_html(index=False) if not bairro_risk_coords_df_cleaned.empty else "<p>Sem dados de risco por bairro para exibir.</p>"
bairro_summary_table_html = ""
if 'bairro_salario_medio' in globals() and 'bairro_idade_media' in globals() and not bairro_salario_medio.empty and not bairro_idade_media.empty:
    bairro_summary_table_html = pd.concat([bairro_salario_medio.set_index('Bairro'), bairro_idade_media.set_index('Bairro')], axis=1).head(10).to_html()
elif 'bairro_salario_medio' in globals() and not bairro_salario_medio.empty:
     bairro_summary_table_html = bairro_salario_medio.head(10).to_html(index=False)
elif 'bairro_idade_media' in globals() and not bairro_idade_media.empty:
     bairro_summary_table_html = bairro_idade_media.head(10).to_html(index=False)
else:
    bairro_summary_table_html = "<p>Sem dados de salário ou idade por bairro para exibir.</p>"


html_dashboard = f"""
<div class='dashboard'>
  <div class='header'>
    <h1> Painel de Monitoramento Epidemiológico</h1>
    <p>Atualizado em {data_atual}</p>
  </div>

  <div class='kpis'>
    <div class='kpi'>
        <h2>{total_entrevistados}</h2>
        <p>Entrevistados</p>
    </div>
    <div class='kpi'>
        <h2>{perc_contaminados:.1f}%</h2>
        <p>Contaminados</p>
    </div>
    <div class='kpi'>
        <h2>R$ {media_salario:,.2f}</h2>
        <p>Média Salarial</p>
    </div>
    <div class='kpi'>
        <h2>{media_idade_texto}</h2>
        <p>Idade Média</p>
    </div>
  </div>

  <div class='cards'>
    <div class='card'>
        <h3> Dados de Risco por Bairro</h3>
        {bairro_risk_table_html}
    </div>

    <div class='card'>
        <h3> Salário Médio e Idade Média por Bairro</h3>
        {bairro_summary_table_html}
    </div>
  </div>
</div>
"""

# ============================================================
# 5️ EXIBIR O DASHBOARD (sem erro)
# ============================================================
display(HTML(estilo_html + html_dashboard))
fig_risco.show()
fig_salario.show()
fig_idade.show()
fig_heat.show()

In [ ]:
#from google.colab import files
#files.download("dashboard_apresentacao.html")

import webbrowser
webbrowser.open("dashboard_apresentacao.html")

In [ ]:
# VERSÃO DE CÓDIGO NÚMERO 1.2

# ============================================================
# DASHBOARD COMPLETO - VISUALIZAÇÃO E ANÁLISE INTERATIVA
# ============================================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
from datetime import datetime

# ============================================================
# CÁLCULOS DE BASE
# ============================================================

# Garante que temos dados básicos
if 'base_df' not in globals() or base_df.empty:
    raise ValueError("⚠️ A base de dados está vazia. Suba ou adicione entrevistados antes de rodar o dashboard.")

# Se não existir bairro_summary_df (caso ainda não tenha sido criado)
if 'bairro_summary_df' not in globals():
    bairro_summary_df = base_df.groupby("Bairro Casa").agg(
        Total_Entrevistados=('CPF', 'count'),
        Total_Contaminados=('Contaminado', 'sum'),
        Porcentagem_Infectados=('Contaminado', 'mean')
    ).reset_index().rename(columns={'Bairro Casa': 'Bairro'})
    bairro_summary_df['Porcentagem_Infectados'] *= 100

# Cria uma base com coordenadas e risco
bairro_risk_coords_df = pd.merge(bairro_summary_df, bairros_coords_df, left_on='Bairro', right_index=True, how='left')
bairro_risk_coords_df['Indice_Risco'] = bairro_risk_coords_df['Porcentagem_Infectados']

# Classificação de máscara com base no risco
def classificar_mascara(risco):
    if risco >= 50:
        return "🚨 Obrigatório"
    elif risco >= 20:
        return "⚠️ Moderado"
    else:
        return "😷 Opcional"

bairro_risk_coords_df['Uso_Mascara'] = bairro_risk_coords_df['Indice_Risco'].apply(classificar_mascara)

# Remove linhas sem coordenadas
bairro_risk_coords_df_cleaned = bairro_risk_coords_df.dropna(subset=['latitude', 'longitude']).copy()

# ============================================================
# INDICADORES GERAIS (KPIs)
# ============================================================
total_entrevistados = len(base_df)
total_contaminados = base_df["Contaminado"].sum()
perc_contaminados = (total_contaminados / total_entrevistados) * 100 if total_entrevistados > 0 else 0
media_salario = base_df["Salario"].mean()

# Idade média
if "Data_Nascimento" in base_df.columns:
    try:
        base_df['Data_Nascimento'] = pd.to_datetime(base_df['Data_Nascimento'])
        base_df['Idade'] = datetime.now().year - base_df['Data_Nascimento'].dt.year
        media_idade = base_df["Idade"].mean()
    except:
        media_idade = None
else:
    media_idade = None

data_atual = datetime.now().strftime("%d/%m/%Y")

# ============================================================
# GRÁFICOS INTERATIVOS
# ============================================================

# 1️⃣ Mapa de risco (com uso de máscara)
fig_risco = px.scatter_mapbox(
    bairro_risk_coords_df_cleaned,
    lat="latitude", lon="longitude",
    color="Indice_Risco",
    size="Total_Contaminados",
    text="Uso_Mascara",
    hover_name="Bairro",
    hover_data={"Indice_Risco": True, "Uso_Mascara": True, "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn_r",
    title="📍 Mapa de Risco e Uso de Máscara por Bairro",
    zoom=11,
    mapbox_style="carto-positron"
)

# 2️⃣ Heatmap de contaminação
fig_heat_cont = px.density_mapbox(
    bairro_risk_coords_df_cleaned,
    lat="latitude", lon="longitude", z="Indice_Risco",
    radius=25,
    color_continuous_scale="Reds",
    title="🔥 Mapa de Calor — Contaminação por Bairro",
    mapbox_style="carto-positron",
    zoom=11
)

# 3️⃣ Densidade populacional (estimada pelo número de entrevistados)
fig_heat_pop = px.density_mapbox(
    bairro_risk_coords_df_cleaned,
    lat="latitude", lon="longitude", z="Total_Pessoas",  # coluna existente
    radius=25,
    center=dict(lat=-23.44, lon=-45.07),
    zoom=10,
    mapbox_style="carto-positron",
    title=" Heatmap de Densidade Populacional (Estimado)"
)

# 4️⃣ Distribuição de idade
fig_idade = px.histogram(
    base_df, x="Idade", nbins=15,
    title="📊 Distribuição de Idades",
    color_discrete_sequence=["#007acc"]
)

# 5️⃣ Salário médio por bairro
bairro_salario_medio = base_df.groupby("Bairro Casa")["Salario"].mean().reset_index().rename(
    columns={"Bairro Casa": "Bairro", "Salario": "Salario_Medio"}
)
fig_salario = px.bar(
    bairro_salario_medio,
    x="Bairro", y="Salario_Medio",
    color="Salario_Medio",
    color_continuous_scale="Blues",
    title="💰 Salário Médio por Bairro"
)
fig_salario.update_layout(xaxis_tickangle=-45)

# ============================================================
# DASHBOARD HTML
# ============================================================

estilo_html = """
<style>
.dashboard {
    font-family: 'Segoe UI', sans-serif;
    background-color: #f7f9fc;
    padding: 20px;
}
.header { text-align: center; margin-bottom: 30px; }
.kpis {
    display: flex; justify-content: space-around;
    flex-wrap: wrap; margin-bottom: 30px;
}
.kpi {
    background: linear-gradient(135deg, #007acc, #00b4d8);
    color: white; padding: 25px; border-radius: 15px;
    text-align: center; width: 22%; min-width: 200px;
    box-shadow: 0 3px 10px rgba(0,0,0,0.2);
}
.kpi h2 { font-size: 2.2em; margin: 0; }
</style>
"""

media_idade_texto = f"{media_idade:.1f}" if media_idade else "N/A"

html_dashboard = f"""
<div class='dashboard'>
  <div class='header'>
    <h1> Painel Epidemiológico de Ubatuba 🧭</h1>
    <p>Atualizado em {data_atual}</p>
  </div>
  <div class='kpis'>
    <div class='kpi'><h2>{total_entrevistados}</h2><p>Entrevistados</p></div>
    <div class='kpi'><h2>{perc_contaminados:.1f}%</h2><p>Contaminados</p></div>
    <div class='kpi'><h2>R$ {media_salario:,.2f}</h2><p>Média Salarial</p></div>
    <div class='kpi'><h2>{media_idade_texto}</h2><p>Idade Média</p></div>
  </div>
</div>
"""

display(HTML(estilo_html + html_dashboard))
fig_risco.show()
fig_heat_cont.show()
fig_heat_pop.show()
fig_salario.show()
fig_idade.show()


# ============================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# >>>> CÓDIGO TEMPORÁRIO PARA GERAR PÁGINA LOCAL HTML <<<<<<<<
# ============================================================
# Este trecho cria um arquivo "dashboard_apresentacao.html" local
# para mostrar aos professores. Pode ser APAGADO depois.
# ============================================================

# ============================================================
# 5️⃣ EXIBIR O DASHBOARD + GERAR ARQUIVO HTML COMPLETO
# ============================================================

from IPython.display import display, HTML
import plotly.io as pio

# Exportar cada gráfico Plotly para HTML isolado (com script embutido)
fig_risco_html = pio.to_html(fig_risco, full_html=False, include_plotlyjs='cdn')
fig_salario_html = pio.to_html(fig_salario, full_html=False, include_plotlyjs=False)
fig_idade_html = pio.to_html(fig_idade, full_html=False, include_plotlyjs=False)
fig_heat_html = pio.to_html(fig_heat, full_html=False, include_plotlyjs=False)
fig_heat_pop_html = pio.to_html(fig_heat_pop, full_html=False, include_plotlyjs=False)

# Construir o HTML completo com os gráficos embutidos
html_final = estilo_html + html_dashboard + f"""
<div class='cards'>
  <div class='card'>
    <h3> Mapa de Risco por Bairro</h3>
    {fig_risco_html}
  </div>
  <div class='card'>
    <h3> Salário Médio por Bairro</h3>
    {fig_salario_html}
  </div>
  <div class='card'>
    <h3> Distribuição de Idades</h3>
    {fig_idade_html}
  </div>
  <div class='card'>
    <h3> Heatmap de Risco (Contaminação)</h3>
    {fig_heat_html}
  </div>
  <div class='card'>
    <h3> Heatmap de Densidade Populacional</h3>
    {fig_heat_pop_html}
  </div>
</div>
"""

# Salvar tudo em um arquivo HTML interativo
with open("dashboard_apresentacao.html", "w", encoding="utf-8") as f:
    f.write(html_final)

# Mostrar dentro do notebook também
display(HTML(html_final))

print("✅ Dashboard salvo em: dashboard_apresentacao.html")


# ============================================================
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# <<<<<< FIM DO CÓDIGO TEMPORÁRIO DE APRESENTAÇÃO <<<<<<<<<<<<<
# ============================================================


In [ ]:
# Calculate the average salary per neighborhood
if 'Bairro Casa' in base_df.columns and 'Salario' in base_df.columns:
    bairro_salario_medio = base_df.groupby('Bairro Casa')['Salario'].mean().reset_index()
    bairro_salario_medio = bairro_salario_medio.rename(columns={'Bairro Casa': 'Bairro', 'Salario': 'Salario_Medio'})
    display(bairro_salario_medio)
else:
    print("Columns 'Bairro Casa' or 'Salario' not found in the DataFrame. Cannot calculate average salary per neighborhood.")

In [ ]:
from google.colab import files
files.download("dashboard_apresentacao.html")

In [ ]:
# Create a base Folium map centered at the defined coordinates and with the specified zoom level
# Use existing ubatuba_coords and zoom_level from previous cells

zoom_level = 11

ubatuba_coords = [-23.4500, -45.0900]
m_new_risk = folium.Map(location=ubatuba_coords, zoom_start=zoom_level)

# Create the list of data points for the new risk heatmap [latitude, longitude, weight]
# Use the cleaned risk data DataFrame and the 'Indice_Risco' as weight
if 'bairro_risk_coords_df_cleaned' in globals() and not bairro_risk_coords_df_cleaned.empty:
    heatmap_points_new_risk = bairro_risk_coords_df_cleaned[['latitude', 'longitude', 'Indice_Risco']].values.tolist()
else:
    print("Não há dados válidos de risco por bairro com coordenadas para gerar o novo heatmap.")
    heatmap_points_new_risk = []

# Adiciona uma HeatMap layer(Camada de calor) a base do mapa usando the generated list of data points
if heatmap_points_new_risk: # Only add heatmap if there are valid points
    HeatMap(heatmap_points_new_risk).add_to(m_new_risk)
else:
    print("Não foi possível gerar o novo heatmap devido à falta de dados.")


# Mostrar o Mapa
display(m_new_risk)

In [ ]:
#display(m_risk)

In [ ]:

display(m)